---

## **9. Hands-On Part 2: Loader Showdown — Web vs CSV**

Aayein ab **do bilkul mukhtalif zariyon (sources)** se data load karte hain aur dekhte hain ke LangChain kis tarah unhein aik hi standard format mein tabdeel karta hai. Yeh wahi jagah hai jahan "Universal Shipping Box" wali misal bilkul fit baithti hai!

# `Required Libraries`

In [5]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.document_loaders import CSVLoader

C:\Users\Muhammad Yahya\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.



# **Load from the Web**

In [7]:
web_loader = WebBaseLoader(

    web_path=["https://en.wikipedia.org/wiki/Natural_language_processing"],

)

web_docs = web_loader.load()


# **Load from a CSV**

In [ ]:
csv_loader = CSVLoader(

    file_path="sample_products.csv",
    encoding="utf-8"    #CSVLoader Encoding Issue: Agar CSV file mein non-ASCII characters hoon,
    ,                   #toh encoding="utf-8" na dene par UnicodeDecodeError aa sakta hai.
)

csv_docs = csv_loader.load()

In [36]:
# --- Dono ka Side-by-Side Muqabla (Comparison) ---
print("LOADER SHOWDOWN: Web vs CSV")
print("=" * 60)

# :<25 ka matlab hai "Text ko left-align karein aur 25 characters ki space dein"
# :>15 ka matlab hai "Text ko right-align karein aur 15 characters ki space dein"
# Yeh terminal/notebook mein ek saaf aur aligned table create karta hai
print(f"{'Feature':<25}{'Web':>15}{'CSV':>15}")
print(f"{'-'*25} {'-'*15}{'-'*15}")
print(f"{'Document':<25}{len(web_docs):>15}{len(csv_docs):>15}")

# .keys() metadata ki tamaam keys ke naam deta hai, list() isey printable banata hai
web_keys = list(web_docs[0].metadata.keys())
csv_keys = list(csv_docs[0].metadata.keys())

print(f"{'MetaData Keys':25}{len(web_keys):>15}{len(csv_keys):>15}")

# Yeh tamaam documents ki average content length (characters count) calculate karta hai
# sum(...) saare lengths ko add karta hai, phir hum total count se divide karte hain
web_avg = sum(len(d.page_content) for d in web_docs) / len(web_docs)
csv_avg = sum(len(d.page_content) for d in csv_docs) / len(web_docs)

# :,.0f formats the number with commas and no decimal places (e.g., 85,432)
print(f"{'Avg content length':<25} {web_avg:>13,.0f}ch {csv_avg:>13,.0f}ch")

print(f"\nWeb metadata keys: {web_keys}")
print(f"CSV metadata keys: {csv_keys}")

LOADER SHOWDOWN: Web vs CSV
Feature                              Web            CSV
------------------------- ------------------------------
Document                               1              8
MetaData Keys                          3              2
Avg content length               55,126ch         1,091ch

Web metadata keys: ['source', 'title', 'language']
CSV metadata keys: ['source', 'row']


# 📊 Code Logic & Concept Breakdown

---

## 1. Code Ka Hissa-War Breakdown (Step-by-Step)

```python
web_avg = sum(len(d.page_content) for d in web_docs) / len(web_docs)

for d in web_docs:Yeh web_docs list ke andar majood har Document object par bari bari loop chalata hai.`len(d.page_content)`:Yeh har document ke text (page_content) ke kul characters ki ginti (length) count karta hai.(len(d.page_content) for d in web_docs):Yeh tamam documents ke character counts ki aik generator/list tayar kar deta hai `(maslan: [85432])`.sum(...):Yeh tamaam documents ke lengths ko aapas mein plus (add) kar deta hai taake total characters mil sakein.`/ len(web_docs)`:Aakhir mein, total characters ko total documents ki tadaad se divide kar diya jata hai taake Average (Ausat) nikal aaye.2. Yeh Concept Kyun Zaroori Hai? (Web vs CSV)Is calculation se LangChain ke do bohot aham behavioral differences samne aate hain:`FeatureWebBaseLoaderCSVLoaderData` StrategyPoore webpage ka text 1 Single Document banata hai.CSV ki har ek Row 1 Alag Document banati hai.Total DocumentsKam hotay hain (amuman 1 ya 2).Zyada hotay hain (jitni rows utne documents).Average LengthBohot Bari `(e.g. 50,000+ ch)`Bohot Chhoti (e.g. 100–300 ch)3. Real-World RAG Pipeline Mein Iski AhmiyatRAG `(Retrieval-Augmented Generation)`banate waqt yeh difference bohot count karta hai:🌐 Web Data: Kyunki web_avg bohot bada hota hai, isey aage chal kar Text Splitter se chhotay chhotay chunks mein todna lazmi hota hai.📊 CSV Data: Kyunki csv_avg pehle se hi chhota hota hai (har row alag document hai), isey amuman direct embedding / vector store mein bheja ja sakta hai.